# Uçuş Gecikmesi Tahmini — 2. Temizlik ve ana tablo
Bu notebook uçuşları temizler, yeni değişkenler üretir ve hava durumuyla birleştirip **modele girecek ana tabloyu** kaydeder.

**Ön koşul:** `00_ucus_veri_kesfi` ve `01_hava_durumu` çalışmış olmalı.

**Çıktı:** Drive `UcusGecikme/processed/flights_2025_clean.parquet` — her satır bir uçuş.

**Sütun adı kuralı:**
- `target_arr_del15` → tahmin edeceğimiz değer (1 = varışta 15 dk'dan fazla gecikti)
- `post_` ile başlayanlar → uçuştan **sonra** bilinen bilgiler. Sadece analiz için tutuyoruz, **modele girmeyecek** (sızıntı).
- `o_` / `d_` ile başlayanlar → kalkış / varış havalimanının o günkü hava durumu

Çalışma zamanı → Tümünü çalıştır (5–10 dakika). Bitince .ipynb olarak indirip gönder.

## 0. Kurulum

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'], check=False)

from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/UcusGecikme')
except ImportError:
    BASE = Path('.')

FLIGHTS = BASE / 'raw' / 'flights_2025'
WEATHER_RAW = BASE / 'raw' / 'weather'
PROC = BASE / 'processed'
OUT = PROC / 'flights_2025_clean.parquet'
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 120)

# 7 milyon satır için DuckDB'yi bellek yerine Colab'ın yerel diskindeki geçici bir dosyada çalıştırıyoruz (RAM yetmezliğine karşı)
work_db = Path('/content/work.duckdb') if BASE.as_posix().startswith('/content') else Path('work.duckdb')
work_db.unlink(missing_ok=True)
con = duckdb.connect(str(work_db))
flights_sql = f"read_csv_auto('{FLIGHTS}/*.csv', union_by_name=true, sample_size=-1)"
log = []  # temizlik günlüğü: (adım, etkilenen satır, açıklama)

Mounted at /content/drive


## 1. Havalimanı tablosunu düzeltme
**a) PBI sorunu:** Uçuş verisinde Palm Beach havalimanının kodu `PBI`, ama havalimanı listesinde (OurAirports) kod **`DJT`** olarak
değişmiş (havalimanının adı değiştirilmiş). Eski kodu listenin `keywords` alanında buluyoruz ve koordinatı oradan alıyoruz.

**b) Uzak istasyonlar:** Hava istasyonu havalimanına **50 km'den uzaksa** o hava durumu havalimanını temsil etmez → "bilinmiyor" sayıyoruz.

In [2]:
airports = pd.read_parquet(PROC / 'airports.parquet')
oa = pd.read_csv(BASE / 'raw' / 'ourairports_airports.csv',
                 usecols=['ident', 'type', 'name', 'latitude_deg', 'longitude_deg', 'iata_code', 'keywords'])
oa = oa[oa['type'].isin(['large_airport', 'medium_airport', 'small_airport'])]

fixed = []
for idx in airports.index[airports['lat'].isna()]:
    code = airports.at[idx, 'code']
    kw = oa['keywords'].fillna('').str.upper().str.split(r'[,\s]+')
    hit = oa[kw.apply(lambda toks: code in toks)]
    if len(hit):
        h = hit.iloc[0]
        airports.loc[idx, ['lat', 'lon', 'ap_name', 'icao']] = [h['latitude_deg'], h['longitude_deg'], h['name'], h['ident']]
        fixed.append(code)
        print(f'{code}: listede yeni kodu "{h["iata_code"]}" — {h["name"]} ({h["latitude_deg"]:.4f}, {h["longitude_deg"]:.4f})')
    else:
        print(f'{code}: listede bulunamadı, hava durumu "bilinmiyor" kalacak')

# Düzeltilen havalimanları için en yakın istasyonu (01 numaralı notebook'taki kuralla) seç
long_pq = PROC / 'ghcnd_2025_us_long.parquet'
if fixed:
    good = con.sql(f'''
        SELECT station FROM '{long_pq}' WHERE q_flag IS NULL AND element IN ('TMAX','PRCP')
        GROUP BY station
        HAVING count(DISTINCT CASE WHEN element='TMAX' THEN date END) >= 300
           AND count(DISTINCT CASE WHEN element='PRCP' THEN date END) >= 300''').df()['station']
    stations = pd.read_fwf(WEATHER_RAW / 'ghcnd-stations.txt', colspecs=[(0, 11), (12, 20), (21, 30), (41, 71)],
                           names=['station', 'st_lat', 'st_lon', 'st_name'], dtype={'station': str})
    cand = stations[stations['station'].isin(set(good))].reset_index(drop=True)

    def haversine_km(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
        a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
        return 6371 * 2 * np.arcsin(np.sqrt(a))

    is_usw = cand['station'].str.startswith('USW').values
    for code in fixed:
        idx = airports.index[airports['code'] == code][0]
        d = haversine_km(airports.at[idx, 'lat'], airports.at[idx, 'lon'], cand['st_lat'].values, cand['st_lon'].values)
        near = np.where(is_usw & (d <= 25))[0]
        i = near[np.argmin(d[near])] if len(near) else int(np.argmin(d))
        airports.loc[idx, ['station', 'st_name', 'distance_km']] = [cand.at[i, 'station'], cand.at[i, 'st_name'], round(float(d[i]), 1)]
        print(f'  → istasyon: {cand.at[i, "station"]} {cand.at[i, "st_name"]} ({d[i]:.1f} km)')

MAX_KM = 50
far = airports.loc[airports['distance_km'] > MAX_KM, ['code', 'city', 'ucus', 'st_name', 'distance_km']]
print(f'\nİstasyonu {MAX_KM} km\'den uzak havalimanları (hava durumu "bilinmiyor" olacak):')
print(far.to_string(index=False))
airports['weather_ok'] = airports['station'].notna() & (airports['distance_km'] <= MAX_KM)
airports.to_parquet(PROC / 'airports.parquet', index=False)


İstasyonu 50 km'den uzak havalimanları (hava durumu "bilinmiyor" olacak):
code              city   ucus          st_name  distance_km
 STX Christiansted, VI 2749.0  CYRIL E KING AP         73.1
 ADK   Adak Island, AK  210.0 SAINT PAUL IS AP        717.2


## 2. Günlük hava durumu tablosu (son hali)
Ham ölçümlerden (01'de kaydedilen) eşleşen istasyonların günlük tablosunu DuckDB ile yeniden kuruyoruz ve birkaç türetilmiş değişken ekliyoruz.

**Eksik değer kararı:** Kar ve rüzgar bazı istasyonlarda hiç ölçülmüyor (ör. güneydeki istasyonlar kar ölçmüyor).
Bunları 0 ile doldurmuyoruz; "bilinmiyor" (boş) bırakıyoruz. Kullanacağımız ağaç tabanlı modeller boş değerle çalışabiliyor.
Sis / fırtına / dolu kodları ise sadece olay olduğunda yazılıyor → yazılmadıysa 0.

In [3]:
st_map = airports.loc[airports['weather_ok'], ['code', 'station']]
con.register('st_map', st_map)
weather = con.sql(f'''
    SELECT m.code AS airport, w.date,
           max(CASE WHEN element='TMAX' THEN value END) / 10 AS tmax_c,
           max(CASE WHEN element='TMIN' THEN value END) / 10 AS tmin_c,
           max(CASE WHEN element='PRCP' THEN value END) / 10 AS prcp_mm,
           max(CASE WHEN element='SNOW' THEN value END)      AS snow_mm,
           max(CASE WHEN element='SNWD' THEN value END)      AS snwd_mm,
           max(CASE WHEN element='AWND' THEN value END) / 10 AS awnd_ms,
           max(CASE WHEN element='WSF2' THEN value END) / 10 AS wsf2_ms,
           max(CASE WHEN element='WT01' THEN 1 ELSE 0 END)   AS fog,
           max(CASE WHEN element='WT03' THEN 1 ELSE 0 END)   AS thunder,
           max(CASE WHEN element='WT05' THEN 1 ELSE 0 END)   AS hail
    FROM '{long_pq}' w JOIN st_map m ON w.station = m.station
    WHERE w.q_flag IS NULL
    GROUP BY ALL''').df()
weather['freezing'] = (weather['tmin_c'] <= 0).astype(float).where(weather['tmin_c'].notna())
weather['heavy_rain'] = (weather['prcp_mm'] >= 10).astype(float).where(weather['prcp_mm'].notna())
weather['snow_day'] = (weather['snow_mm'] > 0).astype(float).where(weather['snow_mm'].notna())
weather['date'] = pd.to_datetime(weather['date'])
print(f'Hava durumu: {len(weather):,} satır, {weather["airport"].nunique()} havalimanı')
print((weather.isna().mean() * 100).round(1).rename('eksik_%').to_string())
weather.to_parquet(PROC / 'weather_daily_2025.parquet', index=False)
WX_COLS = [c for c in weather.columns if c not in ('airport', 'date')]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Hava durumu: 127,399 satır, 350 havalimanı
airport        0.0
date           0.0
tmax_c         0.2
tmin_c         0.3
prcp_mm        0.6
snow_mm       33.1
snwd_mm       36.6
awnd_ms       11.7
wsf2_ms       11.6
fog            0.0
thunder        0.0
hail           0.0
freezing       0.3
heavy_rain     0.6
snow_day      33.1


## 3. Uçuşları okuma ve sütunları düzenleme
- Planlanan saatler sayı olarak tutulmuş: `659` = 06:59, `5` = 00:05 → **saat** (0–23) sütunu üretiyoruz.
- Uçuşun durumu: `tamamlandi` / `iptal` / `yon_degistirdi`. İptal ve yön değiştiren uçuşlar hiç varmadığı için **target'ları yok** → modele girmeyecekler, ama analiz için tabloda tutuyoruz.
- Uçuştan sonra bilinen sütunlara `post_` öneki.

In [4]:
# Bazı aylık dosyalarda sayısal sütunlar METİN olarak okunuyor (ör. "0659" veya "06:59").
# Önce hangi sütunların metin geldiğini görelim, sonra hepsini güvenli şekilde sayıya çevirelim.
types = con.sql(f'DESCRIBE SELECT * FROM {flights_sql}').df()[['column_name', 'column_type']]
text_cols = types.loc[types['column_type'] == 'VARCHAR', 'column_name'].tolist()
print('Metin (VARCHAR) olarak okunan sütunlar:', text_cols)


def num(col, typ='DOUBLE'):
    """Sütunu sayıya çevirir; çevrilemeyen değer boş (NULL) olur. '06:59' gibi saatlerdeki ':' işaretini atar."""
    return f"TRY_CAST(replace(trim(CAST({col} AS VARCHAR)), ':', '') AS {typ})"


con.sql(f'''
CREATE OR REPLACE TABLE flights AS
SELECT
    TRY_CAST(FlightDate AS DATE)            AS flight_date,
    {num('Month', 'INTEGER')}               AS month,
    {num('DayofMonth', 'INTEGER')}          AS day_of_month,
    {num('DayOfWeek', 'INTEGER')}           AS day_of_week,
    Reporting_Airline                       AS airline,
    {num('Flight_Number_Reporting_Airline', 'INTEGER')} AS flight_number,
    Origin                                  AS origin,
    OriginState                             AS origin_state,
    Dest                                    AS dest,
    DestState                               AS dest_state,
    Origin || '-' || Dest                   AS route,
    {num('CRSDepTime', 'INTEGER')}          AS crs_dep_hhmm,
    {num('CRSArrTime', 'INTEGER')}          AS crs_arr_hhmm,
    ({num('CRSDepTime', 'INTEGER')} // 100) % 24 AS dep_hour,
    ({num('CRSArrTime', 'INTEGER')} // 100) % 24 AS arr_hour,
    {num('CRSElapsedTime')}                 AS crs_elapsed_min,
    {num('Distance')}                       AS distance_mi,
    CASE WHEN {num('Cancelled')} = 1 THEN 'iptal'
         WHEN {num('Diverted')} = 1 THEN 'yon_degistirdi'
         ELSE 'tamamlandi' END              AS status,
    {num('ArrDel15', 'INTEGER')}            AS target_arr_del15,
    CancellationCode                        AS post_cancellation_code,
    {num('ArrDelay')}                       AS post_arr_delay_min,
    {num('DepDelay')}                       AS post_dep_delay_min,
    {num('DepDel15', 'INTEGER')}            AS post_dep_del15,
    {num('TaxiOut')}                        AS post_taxi_out_min,
    {num('TaxiIn')}                         AS post_taxi_in_min,
    {num('ActualElapsedTime')}              AS post_actual_elapsed_min,
    {num('AirTime')}                        AS post_air_time_min,
    {num('CarrierDelay')}                   AS post_carrier_delay_min,
    {num('WeatherDelay')}                   AS post_weather_delay_min,
    {num('NASDelay')}                       AS post_nas_delay_min,
    {num('SecurityDelay')}                  AS post_security_delay_min,
    {num('LateAircraftDelay')}              AS post_late_aircraft_delay_min
FROM {flights_sql}''')

bad = con.sql('''SELECT count(*) FILTER (WHERE flight_date IS NULL) AS tarih_okunamayan,
                        count(*) FILTER (WHERE crs_dep_hhmm IS NULL) AS kalkis_saati_okunamayan,
                        count(*) FILTER (WHERE crs_arr_hhmm IS NULL) AS varis_saati_okunamayan
                 FROM flights''').df()
print('Sayıya çevrilemeyen değerler:')
print(bad.to_string(index=False))

n_raw = con.sql('SELECT count(*) FROM flights').fetchone()[0]
log.append(('Ham veri', n_raw, '12 aylık dosya birleştirildi'))
print(f'Ham uçuş: {n_raw:,}')
print(con.sql('''SELECT status, count(*) AS ucus, round(count(*) * 100.0 / sum(count(*)) OVER (), 2) AS yuzde
                 FROM flights GROUP BY 1 ORDER BY 2 DESC''').df().to_string(index=False))
for s, n in con.sql("SELECT status, count(*) FROM flights WHERE status <> 'tamamlandi' GROUP BY 1").fetchall():
    log.append((f'Durum: {s}', n, 'Uçuş varmadı → target yok, modele girmeyecek (analizde tutuldu)'))

print('\nİptal sebepleri (A=havayolu, B=hava durumu, C=hava trafik sistemi, D=güvenlik):')
print(con.sql('''SELECT post_cancellation_code AS sebep, count(*) AS ucus FROM flights
                 WHERE status = 'iptal' GROUP BY 1 ORDER BY 2 DESC''').df().to_string(index=False))

Metin (VARCHAR) olarak okunan sütunlar: ['Reporting_Airline', 'Origin', 'OriginCityName', 'OriginState', 'Dest', 'DestCityName', 'DestState', 'CRSDepTime', 'DepTime', 'WheelsOff', 'WheelsOn', 'CRSArrTime', 'ArrTime', 'CancellationCode']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sayıya çevrilemeyen değerler:
 tarih_okunamayan  kalkis_saati_okunamayan  varis_saati_okunamayan
                0                        0                       0
Ham uçuş: 7,001,619
        status    ucus  yuzde
    tamamlandi 6879485  98.26
         iptal  102876   1.47
yon_degistirdi   19258   0.28

İptal sebepleri (A=havayolu, B=hava durumu, C=hava trafik sistemi, D=güvenlik):
sebep  ucus
    B 63404
    A 21073
    C 18349
    D    50


## 4. Tutarlılık kontrolleri

In [5]:
checks = {
    'Tamamlanan ama target boş': "status = 'tamamlandi' AND target_arr_del15 IS NULL",
    'Target, gecikme dakikasıyla çelişiyor (ArrDel15 ≠ gecikme ≥ 15)':
        "status = 'tamamlandi' AND target_arr_del15 <> CAST(post_arr_delay_min >= 15 AS INTEGER)",
    'Kalkış = varış havalimanı': 'origin = dest',
    'Mesafe ≤ 0 veya boş': 'distance_mi IS NULL OR distance_mi <= 0',
    'Planlanan süre ≤ 0 veya boş': 'crs_elapsed_min IS NULL OR crs_elapsed_min <= 0',
    'Planlanan saat geçersiz (dakika ≥ 60)': '(crs_dep_hhmm % 100) >= 60 OR (crs_arr_hhmm % 100) >= 60',
}
for name, cond in checks.items():
    n = con.sql(f'SELECT count(*) FROM flights WHERE {cond}').fetchone()[0]
    print(f'{name:65s}: {n:,}')
    log.append((name, n, 'kontrol'))

dup = con.sql('''SELECT count(*) FROM (SELECT flight_date, airline, flight_number, origin FROM flights
                 GROUP BY ALL HAVING count(*) > 1)''').fetchone()[0]
print(f'{"Aynı tarih + havayolu + uçuş no + kalkış tekrarı":65s}: {dup:,}')
log.append(('Tekrar eden uçuş', dup, 'kontrol'))

print('\nAşırı gecikmeler (tamamlanan uçuşlarda varış gecikmesi):')
print(con.sql('''SELECT count(*) FILTER (WHERE post_arr_delay_min >= 180) AS "3 saat+",
                        count(*) FILTER (WHERE post_arr_delay_min >= 600) AS "10 saat+",
                        count(*) FILTER (WHERE post_arr_delay_min >= 1440) AS "24 saat+",
                        max(post_arr_delay_min) AS en_uzun_dk
                 FROM flights WHERE status = 'tamamlandi' ''').df().to_string(index=False))
print('→ Bunları silmiyoruz: target "15 dk üstü mü?" sorusu olduğu için uç değerler target\'ı bozmuyor.')

Tamamlanan ama target boş                                        : 1
Target, gecikme dakikasıyla çelişiyor (ArrDel15 ≠ gecikme ≥ 15)  : 0
Kalkış = varış havalimanı                                        : 0
Mesafe ≤ 0 veya boş                                              : 0
Planlanan süre ≤ 0 veya boş                                      : 7
Planlanan saat geçersiz (dakika ≥ 60)                            : 0
Aynı tarih + havayolu + uçuş no + kalkış tekrarı                 : 0

Aşırı gecikmeler (tamamlanan uçuşlarda varış gecikmesi):
 3 saat+  10 saat+  24 saat+  en_uzun_dk
  111405     10152       644      4336.0
→ Bunları silmiyoruz: target "15 dk üstü mü?" sorusu olduğu için uç değerler target'ı bozmuyor.


## 5. Yeni değişkenler (hepsi uçuştan ÖNCE bilinir)
- **Havalimanı yoğunluğu:** O gün, o saatte kalkış havalimanından planlanan kalkış sayısı; varış havalimanına aynı saatte planlanan iniş sayısı.
  Tarifeler önceden belli olduğu için sızıntı değil (iptal olanlar da tarifede vardı, onları da sayıyoruz).
- **Tatile yakınlık:** En yakın ABD resmî tatiline kaç gün var (Şükran Günü, Noel, 4 Temmuz…), en fazla 14.

In [6]:
con.sql('''
CREATE OR REPLACE TABLE flights AS
SELECT *,
       count(*) OVER (PARTITION BY origin, flight_date, dep_hour) AS origin_sched_dep_hour,
       count(*) OVER (PARTITION BY origin, flight_date)           AS origin_sched_dep_day,
       count(*) OVER (PARTITION BY dest, flight_date, arr_hour)   AS dest_sched_arr_hour,
       count(*) OVER (PARTITION BY airline, flight_date)          AS airline_sched_day
FROM flights''')

hol = USFederalHolidayCalendar().holidays('2024-11-01', '2026-02-28')
days = pd.DataFrame({'flight_date': pd.date_range('2025-01-01', '2025-12-31')})
days['days_to_holiday'] = days['flight_date'].apply(lambda d: int(min(abs((d - h).days) for h in hol))).clip(upper=14)
days['is_holiday'] = (days['days_to_holiday'] == 0).astype(int)
days['flight_date'] = days['flight_date'].dt.date
con.register('days', days)
print('2025 resmî tatilleri:', [d.strftime('%d %b') for d in hol if d.year == 2025])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025 resmî tatilleri: ['01 Jan', '20 Jan', '17 Feb', '26 May', '19 Jun', '04 Jul', '01 Sep', '13 Oct', '11 Nov', '27 Nov', '25 Dec']


## 6. Hava durumunu ekleme: kalkış (`o_`) ve varış (`d_`) havalimanı
Not: Burada o günün **gerçekleşen** hava durumunu kullanıyoruz. Gerçek hayatta bunun yerine kalkıştan önce bilinen **hava tahmini** kullanılırdı → sunumda sınırlama olarak belirteceğiz.

In [7]:
weather_j = weather.copy()
weather_j['date'] = weather_j['date'].dt.date
con.register('wx', weather_j)
o_cols = ', '.join(f'ow.{c} AS o_{c}' for c in WX_COLS)
d_cols = ', '.join(f'dw.{c} AS d_{c}' for c in WX_COLS)
con.sql(f'''
CREATE OR REPLACE TABLE final AS
SELECT f.*, dy.days_to_holiday, dy.is_holiday, {o_cols}, {d_cols}
FROM flights f
LEFT JOIN days dy ON f.flight_date = dy.flight_date
LEFT JOIN wx ow ON f.origin = ow.airport AND f.flight_date = ow.date
LEFT JOIN wx dw ON f.dest   = dw.airport AND f.flight_date = dw.date''')

n_final = con.sql('SELECT count(*) FROM final').fetchone()[0]
assert n_final == n_raw, f'Birleştirme satır sayısını değiştirdi! {n_raw:,} → {n_final:,}'
no_o = con.sql('SELECT count(*) FROM final WHERE o_tmax_c IS NULL').fetchone()[0]
no_d = con.sql('SELECT count(*) FROM final WHERE d_tmax_c IS NULL').fetchone()[0]
print(f'Birleştirme sonrası satır: {n_final:,} (değişmedi ✔)')
print(f'Kalkış hava durumu olmayan uçuş: {no_o:,} (%{no_o / n_final * 100:.2f})')
print(f'Varış hava durumu olmayan uçuş: {no_d:,} (%{no_d / n_final * 100:.2f})')
log.append(('Kalkışta hava durumu yok', no_o, 'Uzak istasyon / ölçüm eksik → boş bırakıldı'))
log.append(('Varışta hava durumu yok', no_d, 'Uzak istasyon / ölçüm eksik → boş bırakıldı'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Birleştirme sonrası satır: 7,001,619 (değişmedi ✔)
Kalkış hava durumu olmayan uçuş: 11,179 (%0.16)
Varış hava durumu olmayan uçuş: 11,170 (%0.16)


## 7. Kaydet ve son tabloya bakış

In [8]:
con.sql(f"COPY final TO '{OUT}' (FORMAT parquet, COMPRESSION zstd)")
print(f'Kaydedildi: {OUT}  ({OUT.stat().st_size / 1e6:,.0f} MB)')

n_model = con.sql("SELECT count(*) FROM final WHERE status = 'tamamlandi'").fetchone()[0]
rate = con.sql("SELECT avg(target_arr_del15) FROM final WHERE status = 'tamamlandi'").fetchone()[0]
log.append(('MODEL TABLOSU (tamamlanan uçuşlar)', n_model, f'gecikme oranı %{rate * 100:.1f}'))
cols = con.sql('SELECT * FROM final LIMIT 0').df().columns
print(f'\nSon tablo: {n_final:,} satır × {len(cols)} sütun')
print(f'Modele girecek (tamamlanan) uçuş: {n_model:,}   gecikme oranı: %{rate * 100:.1f}')
print(f'Modelde KULLANILMAYACAK post_ sütunları: {sum(c.startswith("post_") for c in cols)}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Kaydedildi: /content/drive/MyDrive/UcusGecikme/processed/flights_2025_clean.parquet  (223 MB)

Son tablo: 7,001,619 satır × 64 sütun
Modele girecek (tamamlanan) uçuş: 6,879,485   gecikme oranı: %22.3
Modelde KULLANILMAYACAK post_ sütunları: 13


In [9]:
print('TEMİZLİK GÜNLÜĞÜ')
print(pd.DataFrame(log, columns=['adım', 'satır', 'açıklama']).to_string(index=False))

TEMİZLİK GÜNLÜĞÜ
                                                           adım   satır                                                        açıklama
                                                       Ham veri 7001619                                    12 aylık dosya birleştirildi
                                          Durum: yon_degistirdi   19258 Uçuş varmadı → target yok, modele girmeyecek (analizde tutuldu)
                                                   Durum: iptal  102876 Uçuş varmadı → target yok, modele girmeyecek (analizde tutuldu)
                                      Tamamlanan ama target boş       1                                                         kontrol
Target, gecikme dakikasıyla çelişiyor (ArrDel15 ≠ gecikme ≥ 15)       0                                                         kontrol
                                      Kalkış = varış havalimanı       0                                                         kontrol
                               

In [10]:
print('Sütun özeti (tüm tablo):')
s = con.sql('SUMMARIZE final').df()[['column_name', 'column_type', 'approx_unique', 'null_percentage']]
print(s.to_string(index=False))

Sütun özeti (tüm tablo):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                 column_name column_type  approx_unique  null_percentage
                 flight_date        DATE            313             0.00
                       month     INTEGER             13             0.00
                day_of_month     INTEGER             29             0.00
                 day_of_week     INTEGER              7             0.00
                     airline     VARCHAR             13             0.00
               flight_number     INTEGER           6424             0.00
                      origin     VARCHAR            400             0.00
                origin_state     VARCHAR             57             0.00
                        dest     VARCHAR            400             0.00
                  dest_state     VARCHAR             57             0.00
                       route     VARCHAR           7287             0.00
                crs_dep_hhmm     INTEGER           1410             0.00
                crs_arr_hhmm     INTEGER           

## 8. Yeni değişkenlere ilk bakış

In [11]:
print('Planlanan kalkış saatine göre gecikme oranı (tamamlanan uçuşlar):')
print(con.sql('''SELECT dep_hour AS saat, count(*) AS ucus, round(avg(target_arr_del15) * 100, 1) AS gecikme_yuzde
                 FROM final WHERE status = 'tamamlandi' GROUP BY 1 ORDER BY 1''').df().to_string(index=False))

print('\nKalkış havalimanının o saatteki yoğunluğuna göre gecikme oranı:')
print(con.sql('''SELECT CASE WHEN origin_sched_dep_hour <= 5 THEN '1-5' WHEN origin_sched_dep_hour <= 15 THEN '6-15'
                             WHEN origin_sched_dep_hour <= 30 THEN '16-30' WHEN origin_sched_dep_hour <= 50 THEN '31-50'
                             ELSE '50+' END AS saatlik_kalkis,
                        count(*) AS ucus, round(avg(target_arr_del15) * 100, 1) AS gecikme_yuzde
                 FROM final WHERE status = 'tamamlandi' GROUP BY 1 ORDER BY min(origin_sched_dep_hour)''').df().to_string(index=False))

print('\nResmî tatile yakınlığa göre gecikme oranı:')
print(con.sql('''SELECT CASE WHEN days_to_holiday = 0 THEN 'tatil günü' WHEN days_to_holiday <= 3 THEN '1-3 gün'
                             WHEN days_to_holiday <= 7 THEN '4-7 gün' ELSE '8+ gün' END AS tatile_uzaklik,
                        count(*) AS ucus, round(avg(target_arr_del15) * 100, 1) AS gecikme_yuzde
                 FROM final WHERE status = 'tamamlandi' GROUP BY 1 ORDER BY min(days_to_holiday)''').df().to_string(index=False))

Planlanan kalkış saatine göre gecikme oranı (tamamlanan uçuşlar):
 saat   ucus  gecikme_yuzde
    0  11076           16.5
    1   3393           17.2
    2   1299           20.9
    3    732           17.9
    4    298           13.4
    5 194931            8.9
    6 474704           10.1
    7 501217           12.7
    8 478909           14.5
    9 390739           15.8
   10 427414           17.3
   11 431894           19.4
   12 408052           21.1
   13 410959           22.7
   14 397468           25.8
   15 394967           27.9
   16 394921           29.7
   17 432324           31.2
   18 408988           31.8
   19 373468           32.6
   20 303871           31.8
   21 229222           29.1
   22 153112           28.8
   23  55527           20.0

Kalkış havalimanının o saatteki yoğunluğuna göre gecikme oranı:
saatlik_kalkis    ucus  gecikme_yuzde
           1-5 1403006           20.6
          6-15 1604703           20.2
         16-30 1868900           23.2
         31-50 10

## Bitti
**Dosya → İndir → .ipynb** ile indirip gönder. Sonraki adım: keşifsel veri analizi (EDA) — sunumdaki bulgu grafikleri.